## Objective

Todo

## Setup and data loading

### Imports and settings

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

RANDOM_STATE = 201

### File paths

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ACME-HappinessSurvey2020.csv"

### Load and preview the data

In [5]:
raw_df = pd.read_csv(RAW_DATA_PATH)

print(f"Rows: {raw_df.shape[0]}")
print(f"Columns: {raw_df.shape[1]}")

raw_df.head()

Rows: 126
Columns: 7


,Y,X1,X2,X3,X4,X5,X6
0,0,3,3,3,4,2,4
1,0,3,2,3,5,4,3
2,1,5,3,3,3,3,5
3,0,5,4,3,3,3,5
4,0,5,4,3,3,3,5


## Data quality checks

### Expected columns and data types

In [6]:
TARGET_COLUMN = "Y"
FEATURE_COLUMNS = ["X1", "X2", "X3", "X4", "X5", "X6"]

expected_columns = [TARGET_COLUMN, *FEATURE_COLUMNS]

assert not raw_df.empty, "The dataset is empty."
assert raw_df.columns.is_unique, "Column names are not unique."
assert set(raw_df.columns) == set(expected_columns), "Unexpected or missing columns."

raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Y       126 non-null    int64
 1   X1      126 non-null    int64
 2   X2      126 non-null    int64
 3   X3      126 non-null    int64
 4   X4      126 non-null    int64
 5   X5      126 non-null    int64
 6   X6      126 non-null    int64
dtypes: int64(7)
memory usage: 7.0 KB


### Missing values and unique value counts

In [10]:
quality_summary = pd.DataFrame(
    {
        "dtype": raw_df.dtypes,
        "missing_count": raw_df.isna().sum(),
        "missing_pct": raw_df.isna().mean().mul(100).round(1),
        "unique_count": raw_df.nunique(),
    }
)

quality_summary

,dtype,missing_count,missing_pct,unique_count
Y,int64,0,0.0,2
X1,int64,0,0.0,4
X2,int64,0,0.0,5
X3,int64,0,0.0,5
X4,int64,0,0.0,5
X5,int64,0,0.0,5
X6,int64,0,0.0,5


### Values outside the allowed sets

In [11]:
allowed_values = {column: [1, 2, 3, 4, 5] for column in FEATURE_COLUMNS}
allowed_values[TARGET_COLUMN] = [0, 1]

invalid_counts = {}

for column in expected_columns:
    values = raw_df[column]
    invalid_counts[column] = (
        values.notna() & ~values.isin(allowed_values[column])
    ).sum()

pd.Series(invalid_counts, name="invalid_count")

Y     0
X1    0
X2    0
X3    0
X4    0
X5    0
X6    0
Name: invalid_count, dtype: int64

### Repeated rows and answer patterns

An **answer pattern** is one complete combination of scores across X1–X6.
A pattern repeats when at least two rows share those six scores.

The table includes every repeated answer pattern:

- **row_count:** Number of original rows with that pattern.
- **target_count:** Number of distinct happiness labels within that pattern.
  A value of 1 means all labels agree. A value of 2 means both labels occur.

In [18]:
profile_summary = (
    raw_df.groupby(FEATURE_COLUMNS, dropna=False)
    .agg(
        row_count=(TARGET_COLUMN, "size"),
        target_count=(TARGET_COLUMN, "nunique"),
    )
    .reset_index()
)

repeated_answer_patterns = (
    profile_summary.loc[profile_summary["row_count"].gt(1)]
    .sort_values("target_count", ascending=False, kind="stable")
    .reset_index(drop=True)
)

print(f"Exact duplicate rows beyond the first occurrence: {raw_df.duplicated().sum()}")
print(f"Answer patterns that occur at least twice: {len(repeated_answer_patterns)}")
print(
    "Repeated answer patterns with both target labels: "
    f"{repeated_answer_patterns['target_count'].gt(1).sum()}"
)

print("\nRepeated Answer Patterns")

display(repeated_answer_patterns)

Exact duplicate rows beyond the first occurrence: 16
Answer patterns that occur at least twice: 19
Repeated answer patterns with both target labels: 7

Repeated Answer Patterns


,X1,X2,X3,X4,X5,X6,row_count,target_count
0,4,2,3,3,4,4,2,2
1,4,2,4,4,4,4,3,2
2,4,3,3,4,2,4,2,2
3,5,2,3,3,2,5,2,2
4,5,2,4,4,5,5,3,2
5,5,3,3,3,5,3,2,2
6,5,3,3,4,4,5,3,2
7,3,2,4,4,4,5,2,1
8,3,3,1,3,3,4,2,1
9,4,1,3,4,4,4,2,1


## Data split and EDA

## Baseline and model comparison

## Survey question selection

## Final evaluation 

## Conclusions 